In [ ]:
import numpy as np # math library, simulation of random variables
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats # stats library
import seaborn as sns # fancy stats visualization

# Generating Gaussian n samples and visualizing densities

The numpy library can simulate a sample from most usual probability laws (numpy.random). To generate a random sample, the algorithm depends on a sample from a uniform distribution $\mathcal{U}[0,1]$, this is the basic building block. To estimate a uniform distribution, one use a recurrent sequence and the difference between two terms of this sequence mimics the properties of i.i.d sample uniform law. As for all sequence, we need to initialize the sequence with a seed.

The proper way to use a random number generator is to specify the seed: this allow to reproduce the experiments. The code below generate $n$ samples from a standard normal random variable in dimension 2. If you evaluate the cell twice, you will see the same thing. You can also try to change the seed or to not specify it to see what happens.

In [ ]:
n = 100 # Sample size
# initialize a random number generator
rng = np.random.RandomState(2026)
X = rng.normal(size=[n,2]) # random sampling from a standard normal distribution in dimension 2
plt.scatter(X[:,0],X[:,1]) # scatter plot of X

The cells below generate data from multivariate Gaussian distributions with different means and covariances, and visualize the data together with a level set of its density. 

Note that to get the density of a mutlivariate Gaussian random variable, we use the package scipy.stats. 

In [ ]:
def plot_2d_gaussian(mean, cov, ax=None, levels=1, **kwargs):
    """
    Plot the level sets (contours) of a 2D Gaussian distribution.

    Parameters:
    - mean: array-like, shape (2,)
        Mean of the Gaussian.
    - cov: array-like, shape (2, 2)
        Covariance matrix of the Gaussian.
    - ax: matplotlib axis object, optional
        Axis to plot on. If None, a new figure is created.
    - levels: int or array-like, optional
        Number of contour levels or specific levels to draw.
    - **kwargs: additional keyword arguments for contour plotting.

    Returns:
    - ax: matplotlib axis object
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 6))

    # Create grid
    x, y = np.mgrid[-5:5:.05, -5:5:.05]
    pos = np.dstack((x, y))

    # Multivariate normal distribution
    rv = stats.multivariate_normal(mean, cov)

    # Plot contours
    ax.contour(x, y, rv.pdf(pos), levels=levels, **kwargs)
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')
    ax.grid(True)

    return ax


In [ ]:
_, ax = plt.subplots(figsize=(8, 6))

rng = np.random.RandomState(2025)
n = 100 # sample size

mean=[-1,0]
cov=[[1, 0],[0,1]]
X = rng.multivariate_normal(mean,cov,size=[n,2]) # random sampling from a multivariate normal distribution in dimension 2\,
plt.scatter(X[:,0],X[:,1]) # scatter plot of X
plot_2d_gaussian([0,0],[[1,0],[0,1]],ax)

mean2=[2,2]
cov2=[[1, 0.5],[0.5,1]]
Z = rng.multivariate_normal(mean2,cov2,size=[n,2])
plt.scatter(Z[:,0],Z[:,1],color='orange') # scatter plot of X
plot_2d_gaussian(mean2,cov2,ax,colors='orange')

# Generating samples from a Gaussian mixture

Numpy.random doesn't implement a way to sample from Gaussian mixture. Complete the code below to generate an $n$ sample from a Gaussian mixture. The latent variable are also generated, even if they will *not* be given to the EM algorithm. 

In [1]:
def sample_gmm(means, covariances, weights, n_samples,seed=42):
    """
    Generate samples from a GMM with specified means, covariances, and weights.

    Parameters:
    - means: list of arrays, shape (n_components, n_features)
    - covariances: list of arrays, shape (n_components, n_features, n_features)
    - weights: array-like, shape (n_components,)
    - n_samples: int, number of samples to generate

    Returns:
    - samples: array, shape (n_samples, n_features)
    - labels: array, shape (n_samples,), component indices
    """
    rng = np.random.RandomState(seed)
    
    n_components = len(means)
        
    # Normalize weights to sum to 1 (in case they don't already!)
    weights = np.asarray(weights)
    weights = weights / weights.sum()

    # initialize the samples
    samples = np.zeros((n_samples, means[0].shape[0]))

    # draw the latent variable Z_i for each i 
    membership = rng.choice(n_components, size=n_samples, p=weights)

    # draw the corresponding multivariate Gaussian sample 
    
    ### TO BE COMPLETED 

    return samples, membership



## Gaussian mixture in dimension 1 

In [ ]:
K=2

mu1 = np.array([-2])
cov1 = np.array([[0.5]])

mu2 = np.array([1])
cov2 = np.array([[1]])

weights = [0.4, 0.6]

n_samples=500

samples,memberships = sample_gmm([mu1,mu2],[cov1,cov2],weights,n_samples)
sns.histplot(samples,kde=True) # histogram of the sample together with an estimated density

## Gaussian mixture in dimension 2

In [ ]:
K=3

mu1 = np.array([0, 0])
cov1 = np.array([[1, 0.5], [0.5, 1]])

mu2 = np.array([3, 2])
cov2 = np.array([[1, -0.5], [-0.5, 1]])

mu3 = np.array([-2,-3])
cov3 = np.array([[1,0],[0,1]])

weights = [0.3,0.4,0.3]

n_samples=500

samples,memberships = sample_gmm([mu1,mu2,mu3],[cov1,cov2,cov3],weights,n_samples)

_, ax = plt.subplots(figsize=(8, 6))
plt.scatter(samples[:,0],samples[:,1],c = memberships)

# add the covariances 
plot_2d_gaussian(mu1,cov1,ax)
plot_2d_gaussian(mu2,cov2,ax)
plot_2d_gaussian(mu3,cov3,ax)

# Implementation of the EM algorithm 

You can complete the code below to implement the EM algorithm and visualize itermediate iterations. 

In [2]:
def plot_em_iteration(X, means, covariances, responsibilities, iteration):
    """
    visualize the current means, covariances and cluster "membership" 
    """
    plt.figure()
    _, ax = plt.subplots(figsize=(8, 6))

    ax.scatter(X[:, 0], X[:, 1], c=np.argmax(responsibilities, axis=1), alpha=0.6, cmap='viridis')

    # Plot Gaussian contours
    for k in range(len(means)):
        plot_2d_gaussian(means[k], covariances[k], ax, colors=f'C{k}')
        plt.plot(means[k,0],means[k,1],marker='x', color=f'C{k}', markersize=10)

    ax.set_title(f'EM Iteration {iteration}')
    ax.grid(True)


def EM_gmm(X, n_components, max_iter=100, tol=1e-4,iterations_to_display=[],seed=57):
    """
    Run the EM algorithm 

    Parameters:
    - X: data in R^d
    - n_components: number of component in the mixture 
    - max_iter : number of iterations
    - tol : early stopping if the distance between the means is below this value 
    - iterations_to display : a list of intermediate iteration that should be illustrated
    - seed: seed of the random number generator used for the initalization

    """
    
    n_samples, n_features = X.shape

    # should we display something? 
    display = (len(iterations_to_display)> 0)
 
    # Pick randomly some points from the dataset (without replacement) to be the first means
    rng = np.random.RandomState(seed)
    means = X[rng.choice(n_samples, n_components, replace=False)]
    covariances = [np.eye(n_features) for _ in range(n_components)]
    weights = np.ones(n_components) / n_components
    responsibilities = np.zeros((n_samples, n_components))

    if display: 
        # display initialization
        plot_em_iteration(X,means,covariances,np.ones((n_samples,n_components)),"initialization")
    
    for iteration in range(max_iter):

        # E-step: Compute responsibilities
        ### TO BE COMPLETED

        # M-step: Update parameters
        ### TO BE COMPLETED

        # (optionally) illustrate the iteration
        if display & (iteration in iterations_to_display):
            plot_em_iteration(X,means,covariances,responsibilities,iteration)
            
        # Check for convergence
        if iteration > 0 and np.max(np.abs(means - old_means)) < tol:
            plot_em_iteration(X,means,covariances,responsibilities,iteration)
            break
        old_means = means.copy()

        if iteration==max_iter-1:
            plot_em_iteration(X,means,covariances,responsibilities,iteration)
        

    return means, covariances, weights, responsibilities,iteration


Try the EM algorithm on an example. You can explore the sensitivity to the initialization by changing the seed. You can also explore more difficult settings (more components, closer means...) by changing the instance. Or you can try different initializations.

In [ ]:
# sample data from a GMM

K=3
n_samples=1000

mu1 = np.array([0, 0])
cov1 = np.array([[1, 0.5], [0.5, 1]])

mu2 = np.array([3, 2])
cov2 = np.array([[1, -0.5], [-0.5, 1]])

mu3 = np.array([-1,-3])
cov3 = np.array([[1,0],[0,1]])

weights = [0.3,0.4,0.3]

samples,memberships = sample_gmm([mu1,mu2,mu3],[cov1,cov2,cov3],weights,n_samples,2026)

means, covariances, weights, responsibilities,iteration = EM_gmm(samples,3,iterations_to_display=[0,10,50])
print(iteration)

In this notebook, you implemented EM yourself for Gaussian Mixture Models. But if you want to use for some application, you can use the scikit-learn mixture package, see the documentation [here](https://scikit-learn.org/stable/modules/mixture.html).